In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib  # For saving the model

# Step 1: Load the data
data = pd.read_csv("extracted_features.csv")

# Step 2: Split features and labels
X = data.drop(columns=["label"])  # Adjust the column name as needed
y = data["label"]

# Step 3: Split into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Set up the Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# Step 5: Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees
    'max_depth': [10, 15, 20],         # Maximum depth of trees
    'min_samples_split': [2, 5, 10],   # Minimum samples to split an internal node
    'min_samples_leaf': [1, 2, 4],     # Minimum samples per leaf
    'class_weight': [None, 'balanced'],  # Handle class imbalance
}

# Step 6: Set up GridSearchCV to find the best parameters
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Step 7: Fit the model using GridSearchCV
grid_search.fit(X_train, y_train)

# Step 8: Print the best parameters found
print("Best Parameters from GridSearchCV:")
print(grid_search.best_params_)

# Step 9: Train the Random Forest model with the best parameters
best_rf_model = grid_search.best_estimator_

# Step 10: Save the best model
joblib.dump(best_rf_model, 'best_audio_classification_model.pkl')
print("Best model saved as 'best_audio_classification_model.pkl'.")

# Step 11: Make predictions on the test set
y_pred = best_rf_model.predict(X_test)

# Step 12: Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Step 13: Feature importance
importances = best_rf_model.feature_importances_
feature_importances = pd.DataFrame({"Feature": X.columns, "Importance": importances})
print("\nFeature Importances:")
print(feature_importances.sort_values(by="Importance", ascending=False))


Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best Parameters from GridSearchCV:
{'class_weight': 'balanced', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best model saved as 'best_audio_classification_model.pkl'.
Confusion Matrix:
[[254  12  21]
 [  3 262   2]
 [ 14   5 197]]

Classification Report:
              precision    recall  f1-score   support

       Amlou       0.94      0.89      0.91       287
    Ourkimen       0.94      0.98      0.96       267
      Tagala       0.90      0.91      0.90       216

    accuracy                           0.93       770
   macro avg       0.92      0.93      0.92       770
weighted avg       0.93      0.93      0.93       770


Feature Importances:
   Feature  Importance
4        4    0.037274
3        3    0.024393
62      62    0.015034
61      61    0.014773
10      10    0.014063
..     ...         ...
48      48    0.000000
45      45    0.000000
42      42    0.000000
72      

In [33]:
import joblib
import librosa
import numpy as np
import os

# Load the trained model
model = joblib.load('best_audio_classification_model.pkl')

# Define class labels (adjust as needed)
class_labels = ['Amlou', 'Tagala', 'Ourkimen']

# Feature extraction function for audio files
def extract_features(file_path, n_mfcc=13, hop_length=256):
    try:
        audio, sample_rate = librosa.load(file_path, sr=None)
        n_fft = min(len(audio), 512)  # Set n_fft dynamically
        
        # Extract features
        mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
        delta_mfcc = librosa.feature.delta(mfcc)
        delta2_mfcc = librosa.feature.delta(mfcc, order=2)
        mel = librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
        chroma = librosa.feature.chroma_stft(y=audio, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
        spectral_contrast = librosa.feature.spectral_contrast(y=audio, sr=sample_rate, n_fft=n_fft, hop_length=hop_length)
        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(audio), sr=sample_rate)
        
        # Aggregate features (mean)
        mfcc_features = np.mean(mfcc.T, axis=0)
        delta_features = np.mean(delta_mfcc.T, axis=0)
        delta2_features = np.mean(delta2_mfcc.T, axis=0)
        mel_features = np.mean(mel.T, axis=0)
        chroma_features = np.mean(chroma.T, axis=0)
        spectral_features = np.mean(spectral_contrast.T, axis=0)
        tonnetz_features = np.mean(tonnetz.T, axis=0)
        
        return np.hstack((mfcc_features, delta_features, delta2_features, mel_features, chroma_features, spectral_features, tonnetz_features))
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Predict a single audio file
def predict_audio(model, audio_path):
    features = extract_features(audio_path)
    if features is not None:
        features = features.reshape(1, -1)  # Ensure the correct shape
        prediction = model.predict(features)
        
        # Directly use the predicted label (assuming the model outputs the label as a string)
        predicted_label = prediction[0]  # No need to convert to an integer
        return predicted_label
    else:
        return "Error in feature extraction"

# Predict all files in a folder
def predict_audio_folder(model, folder_path):
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_path = os.path.join(folder_path, file)
            predicted_label = predict_audio(model, file_path)
            print(f"File: {file}, Predicted Label: {predicted_label}")

# Main execution
if __name__ == "__main__":
    audio_folder_path = r"C:\Users\dell\Downloads\v1"  # Replace with your folder containing .wav files
    predict_audio_folder(model, audio_folder_path)


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=922
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducin

File: amlou_laila_V1.wav, Predicted Label: Tagala


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=922
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


File: ourkimen_laiala_V1.wav, Predicted Label: Ourkimen
File: tagala_laila_V1.wav, Predicted Label: Tagala


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=907
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [34]:
audio_folder_path = r"C:\Users\dell\Downloads\V3"  # Replace with your folder containing .wav files
predict_audio_folder(model, audio_folder_path)

c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=930
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducin

File: amlou_v3.wav, Predicted Label: Amlou


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


File: ourkimen_v3.wav, Predicted Label: Ourkimen
File: tagala_V3.wav, Predicted Label: Tagala


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=742
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [35]:

audio_folder_path = r"C:\Users\dell\Downloads\V4"  # Replace with your folder containing .wav files
predict_audio_folder(model, audio_folder_path)

c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


File: amlou_v4.wav, Predicted Label: Tagala


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


File: ourkimen_v4.wav, Predicted Label: Amlou
File: tagala_v4.wav, Predicted Label: Amlou


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=921
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [36]:
Qaudio_folder_path = r"C:\Users\dell\Downloads\V2"  # Replace with your folder containing .wav files
predict_audio_folder(model, audio_folder_path)

c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


File: amlou_v4.wav, Predicted Label: Tagala


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


File: ourkimen_v4.wav, Predicted Label: Amlou
File: tagala_v4.wav, Predicted Label: Amlou


c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=921
  warnings.warn(
c:\Users\dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
